In [13]:
from tkinter import *
from tkinter import messagebox

class SoundSpeed:
   
    T_MIN, T_MAX = 2.0, 30.0
    S_MIN, S_MAX = 25.0, 40.0
    D_MIN, D_MAX = 0.0, 8000.0
    
    def __init__(self, T, S, D):
        self.T = T
        self.S = S
        self.D = D

    def in_range(self):
        return (self.T_MIN <= self.T <= self.T_MAX and
                self.S_MIN <= self.S <= self.S_MAX and
                self.D_MIN <= self.D <= self.D_MAX)
       
    def speed(self):
        """음속 c [m/s] 반환"""
        T, S, D = self.T, self.S, self.D
        c = (1448.96 + 4.591*T - 5.304e-2*T**2 + 2.374e-4*T**3 + 1.340*(S-35) + 1.630e-2*D + 1.675e-7*D**2 - 1.025e-2*T*(S-35) - 7.139e-13*T*D**3)
        return c

    def __repr__(self):
        return f"SoundSpeed(T={self.T}, S={self.S}, D={self.D})"


## 함수 선언 부분 ##
def on_calc():
    try:
        T = float(entry_T.get())
        S = float(entry_S.get())
        D = float(entry_D.get())
    except ValueError:
        messagebox.showerror("입력 오류", "T, S, D 는 숫자로 입력해 주세요.")
        return

    ss = SoundSpeed(T, S, D)
    c = ss.speed()
    result.config(text=f"음속 c = {c:.3f} m/s")

    if not ss.in_range():
        messagebox.showwarning("범위 경고",
            "입력이 Mackenzie 공식 유효범위를 벗어났습니다.\n"
            "(T 2~30 °C, S 25~40 PSU, D 0~8000 m)\n결과는 참고용입니다.")


## 메인 코드 부분 ##
window = Tk()
window.title("연습 8 - 해수 음속 계산기 (Mackenzie 1981)")
window.geometry("400x230")

rows = [("수온 T [°C]:", "25.0"), ("염분 S [PSU]:", "35.0"), ("수심 D [m]:", "1000")]
entries = []
for i, (lab, default) in enumerate(rows):
    Label(window, text=lab).grid(row=i, column=0, padx=10, pady=6, sticky="e")
    e = Entry(window, width=12); e.grid(row=i, column=1, padx=10, pady=6)
    e.insert(0, default)
    entries.append(e)
entry_T, entry_S, entry_D = entries

Button(window, text="계산", width=12, command=on_calc).grid(row=3, column=0, columnspan=2, pady=10)
result = Label(window, text="", font=("Consolas", 13))
result.grid(row=4, column=0, columnspan=2)

window.mainloop()

In [17]:
# =====================================================
# 연습문제 9 (중급·해양학): 선형파(미소진폭파) 계산기
# -----------------------------------------------------
# 목표:
#   주기 T 와 수심 h 로부터 선형파 분산관계를 풀어
#   파장 L, 위상속도 c, 군속도 cg 를 구하는 Wave 클래스를 작성.
#   Entry 로 입력받고 Canvas 에 한 파장의 파형(sin)을 그린다.
#
# 분산관계 (g = 9.81):
#   ω = 2π / T
#   ω² = g·k·tanh(k·h)          ← k(파수) 에 대해 비선형 → 반복법으로 해
#   L = 2π / k,   c = L / T,   cg = c/2 · (1 + 2kh / sinh(2kh))
#
# 반복법(뉴턴법) 으로 k 구하기:
#   f(k)  = g·k·tanh(k·h) − ω²
#   f'(k) = g·tanh(k·h) + g·k·h·sech²(k·h)
#   k ← k − f(k)/f'(k)   (수렴할 때까지)
#   초기값: 천해 근사 k0 = ω / sqrt(g·h)
#
# 주의(오류 가능 지점):
#   - math 에는 sech 가 없다. sech(x) = 1/cosh(x) 로 직접 계산.
#   - h 가 매우 크면 tanh(k·h) → 1 (심해), 매우 작으면 천해. 둘 다 잘 동작해야 함.
#   - 0 으로 나누기: T=0 또는 h=0 입력 방지.
#   - 검증값: T=10s, h=1000m(심해) → L≈156.13 m, c≈15.61 m/s
# =====================================================

from tkinter import *
from tkinter import messagebox
import math

G = 9.81


## 클래스 선언 부분 ##
class Wave:
    def __init__(self, period, depth):
        if period <= 0 or depth <= 0:
            raise ValueError("주기 T 와 수심 h 는 0보다 커야 합니다.")
        self.T = period
        self.h = depth

    def wavenumber(self, tol=1e-10, maxit=200):
        """분산관계를 뉴턴법으로 풀어 파수 k 반환"""
        omega = 2 * math.pi / self.T
        h = self.h
        k = omega / math.sqrt(G * h)    # 천해 초기 추정
        # TODO: 뉴턴 반복으로 k 수렴시키기
        #   f  = G*k*tanh(k*h) - omega**2
        #   df = G*tanh(k*h) + G*k*h*(1/cosh(k*h))**2
        #   k  = k - f/df  (|변화량| < tol 이면 종료)
        for _ in range(maxit):
            tanh = math.tanh(k * h)
            sech2 = (1.0 / math.cosh(k * h)) ** 2
            f  = G * k * tanh - omega ** 2
            df = G * tanh + G * k * h * sech2
            k_1 = k - f / df
            if abs(k_1 - k) < tol:  #abs() = 절댓값 함수
                k = k_1
                break
            k=k_1
        return k

    def wavelength(self):
        return 2 * math.pi / self.wavenumber()

    def phase_speed(self):
        return self.wavelength() / self.T

    def group_speed(self):
        k = self.wavenumber()
        c = self.phase_speed()
        # TODO: cg = 0.5*c*(1 + 2*k*h / sinh(2*k*h)) 반환
        return 0.5 * c * (1 + 2 * k * self.h / math.sinh(2 * k * self.h))
        

    def regime(self):
        """심해/천해/중간 판정 (h/L 기준)"""
        ratio = self.h / self.wavelength()
        if ratio > 0.5:
            return "심해파 (deep)"
        elif ratio < 0.05:
            return "천해파 (shallow)"
        else:
            return "중간 수심 (intermediate)"


## 함수 선언 부분 ##
def on_calc():
    try:
        T = float(entry_T.get())
        h = float(entry_h.get())
    except ValueError:
        messagebox.showerror("입력 오류", "T 와 h 는 숫자로 입력해 주세요.")
        return
    try:
        w = Wave(T, h)
    except ValueError as e:
        messagebox.showerror("값 오류", str(e))
        return

    L  = w.wavelength()
    c  = w.phase_speed()
    cg = w.group_speed()
    result.config(text=f"L  = {L:8.3f} m\n"
                       f"c  = {c:8.3f} m/s\n"
                       f"cg = {cg:8.3f} m/s\n"
                       f"유형: {w.regime()}")
    draw_wave(L)


def draw_wave(L):
    """캔버스에 한 파장(2π) 의 사인파를 그린다."""
    canvas.delete("all")
    W = canvas.winfo_width()
    H = canvas.winfo_height()
    if W < 50 or H < 50:
        return
    mid = H / 2
    amp = H * 0.3
    canvas.create_line(0, mid, W, mid, fill="#cccccc")
    pts = []
    for px in range(0, W + 1, 2):
        phase = 2 * math.pi * px / W      # 화면 폭 = 한 파장
        y = mid - amp * math.sin(phase)
        pts.append((px, y))
    for (x1, y1), (x2, y2) in zip(pts[:-1], pts[1:]):
        canvas.create_line(x1, y1, x2, y2, fill="navy", width=2)
    canvas.create_text(W - 8, 12, anchor="e",
                       text=f"한 파장 L = {L:.1f} m", font=("맑은 고딕", 9))


## 메인 코드 부분 ##
window = Tk()
window.title("연습 9 - 선형파 계산기")
window.geometry("460x360")

top = Frame(window); top.pack(pady=8)
Label(top, text="주기 T [s]:").grid(row=0, column=0, padx=5, pady=4, sticky="e")
entry_T = Entry(top, width=10); entry_T.grid(row=0, column=1, padx=5)
entry_T.insert(0, "10.0")
Label(top, text="수심 h [m]:").grid(row=1, column=0, padx=5, pady=4, sticky="e")
entry_h = Entry(top, width=10); entry_h.grid(row=1, column=1, padx=5)
entry_h.insert(0, "1000")
Button(top, text="계산", width=10, command=on_calc).grid(row=0, column=2, rowspan=2, padx=10)

result = Label(window, text="", font=("Consolas", 11), justify=LEFT)
result.pack()

canvas = Canvas(window, bg="white", height=160)
canvas.pack(fill=BOTH, expand=True, padx=10, pady=8)

window.mainloop()

Exception in Tkinter callback
Traceback (most recent call last):
  File "C:\Users\sjlim\anaconda3\Lib\tkinter\__init__.py", line 1968, in __call__
    return self.func(*args)
           ^^^^^^^^^^^^^^^^
  File "C:\Users\sjlim\AppData\Local\Temp\ipykernel_15680\348401087.py", line 101, in on_calc
    L  = w.wavelength()
         ^^^^^^^^^^^^^^
  File "C:\Users\sjlim\AppData\Local\Temp\ipykernel_15680\348401087.py", line 64, in wavelength
    return 2 * math.pi / self.wavenumber()
                         ^^^^^^^^^^^^^^^^^
  File "C:\Users\sjlim\AppData\Local\Temp\ipykernel_15680\348401087.py", line 53, in wavenumber
    sech2 = (1.0 / math.cosh(k * h)) ** 2
                   ^^^^^^^^^^^^^^^^
OverflowError: math range error
Exception in Tkinter callback
Traceback (most recent call last):
  File "C:\Users\sjlim\anaconda3\Lib\tkinter\__init__.py", line 1968, in __call__
    return self.func(*args)
           ^^^^^^^^^^^^^^^^
  File "C:\Users\sjlim\AppData\Local\Temp\ipykernel_15680\348401

In [20]:
# =====================================================
# 연습문제 10 (중급·해양학): 두 지점 거리·방위 계산기
# -----------------------------------------------------
# 목표:
#   위도/경도로 표현된 두 지점 사이의 대권거리(great-circle distance)와
#   초기 방위각(bearing)을 Haversine 공식으로 계산하는 GeoPoint 클래스를 작성.
#   해양 관측 정점 사이 거리, 항해 거리 계산 등에 쓰인다.
#
# Haversine (지구 반지름 R = 6371 km):
#   Δφ = φ2 − φ1,  Δλ = λ2 − λ1   (라디안)
#   a = sin²(Δφ/2) + cosφ1·cosφ2·sin²(Δλ/2)
#   c = 2·atan2(√a, √(1−a))
#   distance = R·c
#
# 초기 방위각(정북 기준 시계방향, 0~360°):
#   y = sinΔλ·cosφ2
#   x = cosφ1·sinφ2 − sinφ1·cosφ2·cosΔλ
#   bearing = (degrees(atan2(y, x)) + 360) mod 360
#
# 주의(오류 가능 지점):
#   - 위경도는 도(degree) 단위 입력 → 삼각함수 전에 math.radians() 로 변환 필수.
#   - atan2(y, x) 의 인자 순서 (y 가 먼저) 를 헷갈리지 말 것.
#   - 음수 경도(서경), 음수 위도(남위) 도 그대로 처리되어야 한다.
#   - 검증값: 부산(35.18,129.08) → 제주(33.51,126.53) ≈ 298.8 km
# =====================================================

from tkinter import *
from tkinter import messagebox
import math

R_EARTH = 6371.0   # km


## 클래스 선언 부분 ##
class GeoPoint:
    """위도(lat)·경도(lon) 한 지점"""

    def __init__(self, lat, lon, name=""):
        self.lat = lat
        self.lon = lon
        self.name = name

    def distance_to(self, other):
        """other 까지의 대권거리 [km]"""
        # TODO: Haversine 공식으로 거리 계산 후 반환
        phi1 = math.radians(self.lat)
        phi2 = math.radians(other.lat)
        dphi = math.radians(other.lat - self.lat)
        dlmb = math.radians(other.lon - self.lon)
        a = (math.sin(dphi / 2) ** 2
             + math.cos(phi1) * math.cos(phi2) * math.sin(dlmb / 2) ** 2)
        c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
        return R_EARTH * c
    
    def bearing_to(self, other):
        """other 로 향하는 초기 방위각 [deg, 0~360]"""
        # TODO: 위 방위각 공식으로 계산 후 반환
        phi1 = math.radians(self.lat)
        phi2 = math.radians(other.lat)
        dlmb = math.radians(other.lon - self.lon)
        y = math.sin(dlmb) * math.cos(phi2)
        x = (math.cos(phi1) * math.sin(phi2)
             - math.sin(phi1) * math.cos(phi2) * math.cos(dlmb))
        return (math.degrees(math.atan2(y, x)) + 360) % 360
        pass

    def __repr__(self):
        return f"GeoPoint({self.lat}, {self.lon}, {self.name!r})"


## 함수 선언 부분 ##
def on_calc():
    try:
        p1 = GeoPoint(float(e_lat1.get()), float(e_lon1.get()), "지점1")
        p2 = GeoPoint(float(e_lat2.get()), float(e_lon2.get()), "지점2")
    except ValueError:
        messagebox.showerror("입력 오류", "위도/경도는 숫자로 입력해 주세요.")
        return

    d = p1.distance_to(p2)
    b = p1.bearing_to(p2)
    result.config(text=f"거리   = {d:9.3f} km\n방위각 = {b:9.2f}°  (정북 기준)")


## 메인 코드 부분 ##
window = Tk()
window.title("연습 10 - 거리·방위 계산기 (Haversine)")
window.geometry("420x240")

Label(window, text="지점 1", font=("맑은 고딕", 10, "bold")).grid(row=0, column=0, columnspan=2, pady=(10, 2))
Label(window, text="위도:").grid(row=1, column=0, sticky="e", padx=5)
e_lat1 = Entry(window, width=10); e_lat1.grid(row=1, column=1); e_lat1.insert(0, "35.18")
Label(window, text="경도:").grid(row=2, column=0, sticky="e", padx=5)
e_lon1 = Entry(window, width=10); e_lon1.grid(row=2, column=1); e_lon1.insert(0, "129.08")

Label(window, text="지점 2", font=("맑은 고딕", 10, "bold")).grid(row=0, column=2, columnspan=2, pady=(10, 2))
Label(window, text="위도:").grid(row=1, column=2, sticky="e", padx=5)
e_lat2 = Entry(window, width=10); e_lat2.grid(row=1, column=3); e_lat2.insert(0, "33.51")
Label(window, text="경도:").grid(row=2, column=2, sticky="e", padx=5)
e_lon2 = Entry(window, width=10); e_lon2.grid(row=2, column=3); e_lon2.insert(0, "126.53")

Button(window, text="계산", width=14, command=on_calc).grid(row=3, column=0, columnspan=4, pady=12)
result = Label(window, text="", font=("Consolas", 12), justify=LEFT)
result.grid(row=4, column=0, columnspan=4)

window.mainloop()

In [23]:
class Current:
    def __init__(self, u, v, name=""):
        self.u = u
        self.v = v
        self.name = name

    def speed(self):
        # TODO: sqrt(u² + v²) 반환
        return math.sqrt(self.u ** 2 + self.v ** 2)

    def direction(self):
        # TODO: (degrees(atan2(u, v)) + 360) % 360 반환
        return (math.degrees(math.atan2(self.u, self.v)) + 360) % 360

    def __add__(self, other):
        # TODO: 성분끼리 더한 새 Current 객체 반환 (name 은 "합성")
        return Current(self.u + other.u, self.v + other.v, name="합성")
        

    def __repr__(self):
        return f"Current(u={self.u}, v={self.v}, name={self.name!r})"


## 함수 선언 부분 ##
def on_calc():
    try:
        c1 = Current(float(e_u1.get()), float(e_v1.get()), "해류1")
        c2 = Current(float(e_u2.get()), float(e_v2.get()), "해류2")
    except ValueError:
        messagebox.showerror("입력 오류", "u, v 는 숫자로 입력해 주세요.")
        return

    total = c1 + c2     # __add__ 호출
    result.config(text=(
        f"{c1.name}: {c1.speed():6.2f} cm/s, {c1.direction():6.1f}°\n"
        f"{c2.name}: {c2.speed():6.2f} cm/s, {c2.direction():6.1f}°\n"
        f"합성  : {total.speed():6.2f} cm/s, {total.direction():6.1f}°"
    ))
    draw_vectors(c1, c2, total)


def draw_vectors(c1, c2, total):
    canvas.delete("all")
    W = canvas.winfo_width()
    H = canvas.winfo_height()
    if W < 50 or H < 50:
        return
    ox, oy = W / 2, H / 2          # 원점(화면 중앙)

    # 축
    canvas.create_line(0, oy, W, oy, fill="#dddddd")   # E-W
    canvas.create_line(ox, 0, ox, H, fill="#dddddd")   # N-S
    canvas.create_text(W - 12, oy - 10, text="E", fill="#999999")
    canvas.create_text(ox + 12, 12, text="N", fill="#999999")

    # 스케일: 최대 성분 크기에 맞춰 자동
    mx = max(abs(c1.u), abs(c1.v), abs(c2.u), abs(c2.v),
             abs(total.u), abs(total.v), 1)
    scale = (min(W, H) * 0.4) / mx

    def arrow(cur, color, width=2):
        x2 = ox + cur.u * scale
        y2 = oy - cur.v * scale     # 북(+v)을 위로
        canvas.create_line(ox, oy, x2, y2, fill=color, width=width,
                           arrow=LAST, arrowshape=(12, 14, 5))

    arrow(c1, "blue")
    arrow(c2, "green")
    arrow(total, "red", width=3)


## 메인 코드 부분 ##
window = Tk()
window.title("연습 11 - 해류 벡터 합성기")
window.geometry("460x460")

top = Frame(window); top.pack(pady=8)
Label(top, text="해류1  u:").grid(row=0, column=0, sticky="e")
e_u1 = Entry(top, width=7); e_u1.grid(row=0, column=1); e_u1.insert(0, "30")
Label(top, text="v:").grid(row=0, column=2, sticky="e")
e_v1 = Entry(top, width=7); e_v1.grid(row=0, column=3); e_v1.insert(0, "40")

Label(top, text="해류2  u:").grid(row=1, column=0, sticky="e")
e_u2 = Entry(top, width=7); e_u2.grid(row=1, column=1); e_u2.insert(0, "-20")
Label(top, text="v:").grid(row=1, column=2, sticky="e")
e_v2 = Entry(top, width=7); e_v2.grid(row=1, column=3); e_v2.insert(0, "10")

Button(top, text="합성", width=8, command=on_calc).grid(row=0, column=4, rowspan=2, padx=10)

result = Label(window, text="", font=("Consolas", 11), justify=LEFT)
result.pack()

canvas = Canvas(window, bg="white")
canvas.pack(fill=BOTH, expand=True, padx=10, pady=8)

window.mainloop()

In [ ]:
class Population:
    def __init__(self, N0=5.0, K=1000.0, r=0.5, dt=0.1):
        self.N = N0          # 현재 개체수
        self.K = K           # 환경수용력
        self.r = r           # 성장률
        self.dt = dt
        self.history = [N0]  # 시계열 기록

    def step(self):
        """한 시간 스텝 전진 (로지스틱)"""
        # TODO: dN = r*N*(1 - N/K)*dt 계산 후 self.N 갱신
        # TODO: self.history 에 새 N 추가
        pass

    def reset(self, N0=5.0):
        self.N = N0
        self.history = [N0]


## 함수 선언 부분 ##
running = False
after_id = None


def tick():
    global after_id
    pop.r = scale_r.get()          # 슬라이더에서 성장률 읽기
    pop.step()
    draw()
    if running:
        after_id = window.after(60, tick)


def on_start():
    global running
    if running:
        return                     # 중복 시작 방지
    running = True
    tick()


def on_pause():
    global running
    running = False


def on_reset():
    global running
    running = False
    pop.reset()
    draw()


def draw():
    canvas.delete("all")
    W = canvas.winfo_width()
    H = canvas.winfo_height()
    if W < 50 or H < 50:
        return
    pad = 40
    hist = pop.history
    n = len(hist)

    # K 수용력 선
    yK = pad
    canvas.create_line(pad, yK, W - pad, yK, fill="#e0a0a0", dash=(4, 2))
    canvas.create_text(W - pad, yK - 8, anchor="e",
                       text=f"K = {pop.K:.0f}", fill="#c06060", font=("맑은 고딕", 9))

    def to_px(i, N):
        x = pad + (i / max(n - 1, 1)) * (W - 2 * pad)
        y = (H - pad) - (N / pop.K) * (H - 2 * pad)
        return x, y

    # 곡선
    pts = [to_px(i, N) for i, N in enumerate(hist)]
    for (x1, y1), (x2, y2) in zip(pts[:-1], pts[1:]):
        canvas.create_line(x1, y1, x2, y2, fill="seagreen", width=2)

    canvas.create_text(pad + 4, pad - 20, anchor="w",
                       text=f"N = {pop.N:8.1f}    r = {pop.r:.2f}    step = {n-1}",
                       font=("Consolas", 10))


## 메인 코드 부분 ##
pop = Population()

window = Tk()
window.title("연습 12 - 플랑크톤 개체군 성장")
window.geometry("560x420")

top = Frame(window); top.pack(fill=X, pady=6)
Button(top, text="시작", width=7, command=on_start).pack(side=LEFT, padx=4)
Button(top, text="정지", width=7, command=on_pause).pack(side=LEFT, padx=4)
Button(top, text="리셋", width=7, command=on_reset).pack(side=LEFT, padx=4)

Label(top, text="성장률 r:").pack(side=LEFT, padx=(20, 2))
scale_r = Scale(top, from_=0.0, to=2.0, resolution=0.05,
                orient=HORIZONTAL, length=180)
scale_r.set(0.5)
scale_r.pack(side=LEFT)

canvas = Canvas(window, bg="white")
canvas.pack(fill=BOTH, expand=True, padx=10, pady=8)

draw()
window.mainloop()